In [145]:
import requests
import json
import re
import pandas as pd
import numpy as np

def get_uniprot(accession):
  '''
  define http_function to get the data from Uniprot API
  '''
  endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
  http_function = requests.get
  http_args = {'params': {'accessions': accession}}
  return http_function(endpoint, **http_args)

def uniprot_parse_response(resp: dict):
  '''
  parse response from Uniprot and output
  organism, geneInfo, sequenceInfo, type

  do not forget to include error handling (e.g. for key errors)
  '''
  try:
    data = resp.json()

    entry = data['results'][0]
    return {
        'organism': entry.get('organism', {}).get('scientificName'),
        'geneInfo': entry.get('genes'),
        'sequenceInfo': entry.get('sequence'),
        'type': entry.get('entryType')
        }
  except Exception as e:
    return e

def get_ensembl(id):
  '''
  define http_function to get the data from Ensembl API
  '''
  endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
  http_function = requests.get
  http_args = {"headers": {"Content-Type": "application/json"}}
  return http_function(endpoint, **http_args)

def ensembl_parse_response(resp: dict):
  '''
  parse Ensembl response and output
  object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

  do not forget to include error handling (e.g. for key errors)
  '''
  try:
    data = resp.json()
    fields = [
        'object_type', 'species', 'assembly_name', 'biotype',
        'display_name', 'id', 'db_type', 'description', 'source',
        'canonical_transcript'
    ]
    return {field: data.get(field) for field in fields}
  except Exception as e:
    return e

def main(ids: list):
  '''
  Function that iterates over all the provided IDs and parses them into dict,
  transforms into pandas.DataFrame, and return it
  {ID : info from parse_response(), ...}

  If ID is incorrect, it should return {ID : error message}
  '''
  uniprot_pattern = r'^[A-Z][0-9][A-Z0-9]+'
  ensembl_pattern = r'^ENS[A-Z]*[0-9]+'

  output = {}

  for entry_id in ids:
      if re.match(uniprot_pattern, entry_id):
          res = get_uniprot(entry_id)
          if res.status_code == 200:
              output[entry_id] = uniprot_parse_response(res)
          else:
              output[entry_id] = f"error:{res.status_code}"

      elif re.match(ensembl_pattern, entry_id):
          res = get_ensembl(entry_id)
          if res.status_code == 200:
              output[entry_id] = ensembl_parse_response(res)
          else:
              output[entry_id] = f"error:{res.status_code}"

      else:
          output[entry_id] = "error:unknown database"

  return pd.DataFrame(output)

In [136]:
get_uniprot('P11473')


<Response [200]>

In [137]:
get_uniprot('helloworld')


<Response [400]>

In [138]:
get_uniprot('helloworld').json()


{'url': 'http://rest.uniprot.org/uniprotkb/accessions',
 'messages': ["Accession 'helloworld' has invalid format. It should be a valid UniProtKB accession with optional sequence range e.g. P12345[10-20]."]}

In [139]:
uniprot_parse_response(get_uniprot('P11473'))


{'organism': 'Homo sapiens',
 'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
      'source': 'HGNC',
      'id': 'HGNC:12679'}],
    'value': 'VDR'},
   'synonyms': [{'value': 'NR1I1'}]}],
 'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
  'length': 427,
  'molWeight': 48289,
  'crc64': 'F95F300D042C4CB7',
  'md5': '0D963ACD4A34674368324EE026023597'},
 'type': 'UniProtKB reviewed (Swiss-Prot)'}

In [140]:
get_ensembl('ENSMUSG00000041147')


<Response [200]>

In [141]:
get_ensembl('helloworld')


<Response [400]>

In [142]:
get_ensembl('helloworld').json()


{'error': "ID 'helloworld' not found"}

In [143]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))


{'object_type': 'Gene',
 'species': 'mus_musculus',
 'assembly_name': 'GRCm39',
 'biotype': 'protein_coding',
 'display_name': 'Brca2',
 'id': 'ENSMUSG00000041147',
 'db_type': 'core',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'source': 'ensembl_havana',
 'canonical_transcript': 'ENSMUST00000044620.11'}

In [146]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])


,P11473,Q91XI3,hello,ENSG00000157764,ENSG00000139618
organism,Homo sapiens,Ictidomys tridecemlineatus,error:unknown database,NaN,NaN
geneInfo,[{'geneName': {'evidences': [{'evidenceCode': ...,[{'geneName': {'value': 'INS'}}],error:unknown database,NaN,NaN
sequenceInfo,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,error:unknown database,NaN,NaN
type,UniProtKB reviewed (Swiss-Prot),UniProtKB reviewed (Swiss-Prot),error:unknown database,NaN,NaN
object_type,NaN,NaN,error:unknown database,Gene,Gene
species,NaN,NaN,error:unknown database,homo_sapiens,homo_sapiens
assembly_name,NaN,NaN,error:unknown database,GRCh38,GRCh38
biotype,NaN,NaN,error:unknown database,protein_coding,protein_coding
display_name,NaN,NaN,error:unknown database,BRAF,BRCA2
id,NaN,NaN,error:unknown database,ENSG00000157764,ENSG00000139618
